Cell 1: Install Dependencies

In [1]:
!pip install -q transformers torch torchvision scikit-learn pillow

Cell 2: Import Libraries

In [2]:
import torch
import torchvision.models as models
import torchvision.transforms as transforms
from transformers import CLIPProcessor, CLIPModel
from PIL import Image
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
from google.colab import files

Cell 3: Upload Two Images

In [3]:
uploaded = files.upload()

Saving plateau.jpg to plateau.jpg
Saving tree.jpg to tree.jpg


Cell 4: Load Images

In [ ]:
image1_path = "tree.jpg"
image2_path = "plateau.jpg"

image1 = Image.open(image1_path).convert("RGB")
image2 = Image.open(image2_path).convert("RGB")

display(image1)
display(image2)

Cell 5: Load CLIP Model

In [5]:
device = "cuda" if torch.cuda.is_available() else "cpu"

clip_model = CLIPModel.from_pretrained(
    "openai/clip-vit-base-patch32"
).to(device)

clip_processor = CLIPProcessor.from_pretrained(
    "openai/clip-vit-base-patch32"
)

print("CLIP model loaded.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/4.19k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

...

vocab.json:   0%|          | 0.00/862k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/525k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.22M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

CLIP model loaded.


Cell 6: Generate CLIP Embeddings

In [14]:
def get_clip_embedding(image):
    inputs = clip_processor(images=image, return_tensors="pt")

    pixel_values = inputs["pixel_values"].to(device)

    with torch.no_grad():
        outputs = clip_model.vision_model(pixel_values=pixel_values)

        # pooled output
        features = outputs.pooler_output

    # Normalize
    features = features / features.norm(p=2, dim=-1, keepdim=True)

    return features.squeeze().cpu().numpy()


clip_emb1 = get_clip_embedding(image1)
clip_emb2 = get_clip_embedding(image2)

print(clip_emb1.shape)

(768,)


Cell 7: Print CLIP Embeddings

In [15]:
np.set_printoptions(threshold=np.inf)

print("Image 1 CLIP Embedding:")
print(clip_emb1)

print("\nImage 2 CLIP Embedding:")
print(clip_emb2)

Image 1 CLIP Embedding:
[ 4.92586531e-02 -1.13221537e-02 -5.66399004e-03  6.85284566e-03
 -3.29239815e-02 -1.59507850e-03  1.59056783e-02 -9.65937078e-02
  5.48480004e-02 -3.04335002e-02 -1.82488654e-02  9.02595837e-03
  3.79017591e-02  3.56136598e-02 -4.50161323e-02  7.70633444e-02
 -7.70589570e-03  5.09471521e-02 -1.95509158e-02 -1.57118542e-03
  1.95843149e-02 -2.08694581e-03 -6.51000021e-03  4.10066284e-02
 -3.30691449e-02  1.15837073e-02 -3.53476987e-03 -1.01394281e-02
 -2.09627207e-02  4.75698598e-02 -1.17369667e-01 -1.48390187e-02
 -1.41370641e-02  3.18868719e-02 -2.01804331e-04  8.39744806e-02
  2.57339026e-03 -6.95575681e-03  2.86764046e-03 -2.26466265e-02
  1.40404981e-03 -8.83445889e-03 -1.80268884e-02  1.37028834e-02
 -3.38869840e-02 -2.58303504e-03  2.42031571e-02 -6.89012231e-04
 -2.38073543e-02  3.22350636e-02  2.07626037e-02  3.37746255e-02
  4.88339141e-02  3.85930464e-02  4.58390713e-02  1.49589004e-02
  1.65291447e-02  5.80918370e-03  5.26272319e-02  3.80341262e-02
 

Cell 8: Compute CLIP Similarity

In [16]:
clip_similarity = cosine_similarity(
    clip_emb1.reshape(1,-1),
    clip_emb2.reshape(1,-1)
)[0][0]

print("CLIP Similarity:", clip_similarity)
print("CLIP Similarity Percentage:", round(clip_similarity*100,2),"%")

CLIP Similarity: 0.5377177
CLIP Similarity Percentage: 53.77 %


Cell 9: Load ResNet50  # now using resnet

In [17]:
resnet = models.resnet50(
    weights=models.ResNet50_Weights.DEFAULT
)

# Remove classification layer
resnet = torch.nn.Sequential(*list(resnet.children())[:-1])

resnet = resnet.to(device)
resnet.eval()

print("ResNet50 loaded.")

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:01<00:00, 100MB/s] 


ResNet50 loaded.


Cell 10: Define Image Transform

In [18]:
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )
])

Cell 11: Generate ResNet50 Embeddings

In [19]:
def get_resnet_embedding(image):
    img_tensor = transform(image).unsqueeze(0).to(device)

    with torch.no_grad():
        embedding = resnet(img_tensor)

    embedding = embedding.squeeze().cpu().numpy()

    # Normalize
    embedding = embedding / np.linalg.norm(embedding)

    return embedding

resnet_emb1 = get_resnet_embedding(image1)
resnet_emb2 = get_resnet_embedding(image2)

print("ResNet50 Embedding Shape:", resnet_emb1.shape)

ResNet50 Embedding Shape: (2048,)


Cell 12: Print ResNet50 Embeddings

In [20]:
print("Image 1 ResNet50 Embedding:")
print(resnet_emb1)

print("\nImage 2 ResNet50 Embedding:")
print(resnet_emb2)

Image 1 ResNet50 Embedding:
[0.00000000e+00 1.57897605e-03 8.13662354e-03 0.00000000e+00
 0.00000000e+00 8.63959081e-03 1.18481785e-01 0.00000000e+00
 1.96351893e-02 6.20769896e-03 3.36805708e-03 2.29767673e-02
 3.63287772e-03 2.43124552e-03 0.00000000e+00 4.14738618e-03
 4.23331000e-02 4.75308625e-03 0.00000000e+00 0.00000000e+00
 3.97695461e-03 5.68332965e-04 5.39252581e-03 1.09903231e-01
 6.85050254e-05 0.00000000e+00 2.68592080e-03 0.00000000e+00
 1.35222601e-03 6.93521788e-03 0.00000000e+00 0.00000000e+00
 0.00000000e+00 2.57411296e-03 3.10355499e-02 5.84296659e-02
 1.59569943e-04 2.27482580e-02 0.00000000e+00 5.87349897e-03
 9.85993166e-03 7.31172087e-03 8.81828077e-04 0.00000000e+00
 5.70235848e-02 8.00009351e-03 0.00000000e+00 0.00000000e+00
 1.06032984e-03 0.00000000e+00 0.00000000e+00 0.00000000e+00
 0.00000000e+00 0.00000000e+00 9.74095019e-04 1.41059374e-02
 1.43503270e-03 8.95494959e-05 2.13619601e-03 2.05013324e-02
 1.04697386e-03 3.55015160e-03 1.21522625e-03 0.00000000e

Cell 13: Compute ResNet50 Similarity

In [21]:
resnet_similarity = cosine_similarity(
    resnet_emb1.reshape(1,-1),
    resnet_emb2.reshape(1,-1)
)[0][0]

print("ResNet50 Similarity:", resnet_similarity)
print("ResNet50 Similarity Percentage:", round(resnet_similarity*100,2),"%")

ResNet50 Similarity: 0.27208123
ResNet50 Similarity Percentage: 27.21 %


Cell 14: Final Comparison

In [22]:
print("="*50)
print("FINAL RESULTS")
print("="*50)

print(f"CLIP Similarity     : {clip_similarity:.4f}")
print(f"ResNet50 Similarity : {resnet_similarity:.4f}")

print()

print(f"CLIP Percentage     : {clip_similarity*100:.2f}%")
print(f"ResNet50 Percentage : {resnet_similarity*100:.2f}%")

FINAL RESULTS
CLIP Similarity     : 0.5377
ResNet50 Similarity : 0.2721

CLIP Percentage     : 53.77%
ResNet50 Percentage : 27.21%


### Conclusion

The similarity results indicate that the two images are visually different. ResNet50 produced a low similarity score (27.21%) because it primarily captures low-level visual features such as shapes, textures, and colors, which differ significantly between the images. In contrast, CLIP yielded a higher similarity score (53.77%) as it captures high-level semantic information and recognizes that both images represent outdoor landscape scenes. This demonstrates that CLIP is more effective for semantic similarity, whereas ResNet50 is better suited for measuring visual similarity.